# 固定数据划分

**负责人：** 4 号（CAO Taihe）　**阶段：** 1 数据准备

生成全组唯一的一套数据划分，保证所有模型的结果可以比较。划分方法和代码设计来自 4 号的交付（`Member4_file/`）；`src/splitting.py` 按他的说明和测试补齐。

## 任务

标"（选做）"的任务在必做部分完成后再做。

- [x] 和 2 号确认清洗数据的列和 `group_id` 的含义
- [x] 用 `StratifiedGroupKFold` 按 `group_id` 分组：先分出约 20% 测试集，再把剩下的分成 5 折，同一组的推文不能跨训练/测试或分到不同折
- [x] 检查每一折的类别比例
- [x] 保存前运行 `check_splits(splits, clean)`（`validate_splits` 还会检查每个部分都有三个类别）

## 产出

- `splits/folds_v1.csv`，列为 `id, group_id, split, fold`（测试集 fold = -1）

In [ ]:
import sys
from pathlib import Path

# 向上查找仓库根目录（含 src/config.py 的文件夹），在任何子文件夹里打开都能导入 src
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists()), None)
if ROOT is None:
    raise RuntimeError("找不到仓库根目录：请在 clone 下来的仓库里启动 Jupyter（Colab 先 %cd 进仓库）")
sys.path.insert(0, str(ROOT))

from src.config import DATA_ROOT, FIGURES_DIR, LABELS, SEED                 # 路径和常量
from src.data import load_clean, load_raw, load_splits                      # 读取数据和划分
from src.evaluate import evaluate, new_run_id, plot_confusion, save_result  # 统一评分
from src.cache import load_embedding, save_embedding, save_preds            # 嵌入和预测结果的读写

print("仓库根目录:", ROOT)
print("DATA_ROOT:", DATA_ROOT)

## 生成并检查划分

`write_splits` 生成划分并保存；文件已存在时只允许内容完全相同。`split_summary` 给出报告表 5 的数字。

In [ ]:
from src.splitting import split_summary, validate_splits, write_splits

clean = load_clean()
splits = write_splits(clean)
validate_splits(load_splits(clean=clean), clean)
split_summary(clean, splits)